# NMT-Net Enhanced — Final Results & Analysis

This notebook loads the experiment CSV files from `../results/` and generates the final comparison tables and presentation-ready graphs for the Enhanced NMT-Net project.

**Models:** LSTM Baseline vs LSTM + Task Embedding  
**Forgetting evaluation:** sequential training across 10 task groups

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

RESULTS = Path('../results')
FIGURES = RESULTS / 'figures'
FIGURES.mkdir(exist_ok=True)

plt.rcParams['figure.figsize'] = (9, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['figure.dpi'] = 120

print('Results folder:', RESULTS.resolve())
print('Figures folder:', FIGURES.resolve())

## 1. Load result CSV files

In [ ]:
lstm_metrics = pd.read_csv(RESULTS / 'lstm_baseline_metrics.csv')
task_metrics = pd.read_csv(RESULTS / 'task_embedding_metrics.csv')
forget_stage = pd.read_csv(RESULTS / 'forgetting_stage_metrics.csv')
forget_tasks = pd.read_csv(RESULTS / 'forgetting_task_summary.csv')
forget_curve = pd.read_csv(RESULTS / 'forgetting_curve.csv')

print('LSTM metrics:', lstm_metrics.shape)
print('Task embedding metrics:', task_metrics.shape)
print('Forgetting stage metrics:', forget_stage.shape)
print('Forgetting task summary:', forget_tasks.shape)
print('Forgetting curve:', forget_curve.shape)

In [ ]:
print('LSTM columns:', list(lstm_metrics.columns))
print('Task embedding columns:', list(task_metrics.columns))
print('Forgetting stage columns:', list(forget_stage.columns))
print('Forgetting task columns:', list(forget_tasks.columns))
print('Forgetting curve columns:', list(forget_curve.columns))

## 2. Final model comparison table

In [ ]:
# Expected metric names from the completed experiments.
# The helper below also tolerates common capitalization differences.
def find_col(df, candidates):
    normalized = {str(c).strip().lower(): c for c in df.columns}
    for name in candidates:
        if name.lower() in normalized:
            return normalized[name.lower()]
    return None

metric_candidates = {
    'MAE': ['mae'],
    'RMSE': ['rmse'],
    'MAPE (%)': ['mape', 'mape (%)'],
    'sMAPE (%)': ['smape', 'smape (%)'],
    'R²': ['r2', 'r²']
}

def extract_metrics(df):
    row = df.iloc[0]
    result = {}
    for label, candidates in metric_candidates.items():
        col = find_col(df, candidates)
        result[label] = float(row[col]) if col is not None else np.nan
    return result

comparison = pd.DataFrame([
    {'Model': 'LSTM Baseline', **extract_metrics(lstm_metrics)},
    {'Model': 'LSTM + Task Embedding', **extract_metrics(task_metrics)}
])

display(comparison.round(4))
comparison.to_csv(RESULTS / 'final_model_comparison.csv', index=False)
comparison

### Metric comparison chart

In [ ]:
plot_metrics = ['MAE', 'RMSE', 'MAPE (%)', 'sMAPE (%)', 'R²']
fig, axes = plt.subplots(1, 1, figsize=(10, 5))
x = np.arange(len(plot_metrics))
width = 0.35
axes.bar(x - width/2, comparison.loc[0, plot_metrics].astype(float), width, label='LSTM Baseline')
axes.bar(x + width/2, comparison.loc[1, plot_metrics].astype(float), width, label='LSTM + Task Embedding')
axes.set_xticks(x)
axes.set_xticklabels(plot_metrics)
axes.set_ylabel('Metric value')
axes.set_title('LSTM Baseline vs LSTM + Task Embedding')
axes.legend()
plt.tight_layout()
plt.savefig(FIGURES / 'model_comparison_all_metrics.png', bbox_inches='tight')
plt.show()

## 3. Individual model metric charts

In [ ]:
for metric in plot_metrics:
    fig, ax = plt.subplots(figsize=(7, 5))
    values = comparison[metric].astype(float)
    bars = ax.bar(comparison['Model'], values)
    ax.set_title(f'{metric}: Model Comparison')
    ax.set_ylabel(metric)
    ax.tick_params(axis='x', rotation=15)
    for bar, value in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(), f'{value:.4f}', ha='center', va='bottom')
    plt.tight_layout()
    filename = metric.replace('²', '2').replace('%', 'percent').replace(' ', '_') + '_comparison.png'
    plt.savefig(FIGURES / filename, bbox_inches='tight')
    plt.show()

## 4. Sequential catastrophic-forgetting results

The experiment evaluates the model after sequentially introducing groups of tasks. A positive forgetting value means a task's current RMSE is worse than its best previously observed RMSE.

In [ ]:
display(forget_stage)

stage_col = find_col(forget_stage, ['stage'])
seen_col = find_col(forget_stage, ['seen_tasks', 'seen tasks'])
rmse_col = find_col(forget_stage, ['rmse'])
mean_f_col = find_col(forget_stage, ['mean_forgetting', 'mean forgetting'])
max_f_col = find_col(forget_stage, ['max_forgetting', 'max forgetting'])
count_col = find_col(forget_stage, ['forgotten_tasks', 'forgotten tasks'])
deg_col = find_col(forget_stage, ['mean_degradation', 'mean degradation'])

print('Detected columns:')
print({'stage': stage_col, 'seen_tasks': seen_col, 'rmse': rmse_col,
       'mean_forgetting': mean_f_col, 'max_forgetting': max_f_col,
       'forgotten_tasks': count_col, 'mean_degradation': deg_col})

### 4.1 Mean forgetting curve

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(forget_stage[stage_col], forget_stage[mean_f_col], marker='o')
ax.set_xlabel('Sequential Stage')
ax.set_ylabel('Mean Forgetting (RMSE increase)')
ax.set_title('Mean Catastrophic Forgetting Across Sequential Stages')
plt.tight_layout()
plt.savefig(FIGURES / 'mean_forgetting_curve.png', bbox_inches='tight')
plt.show()

### 4.2 Maximum forgetting curve

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(forget_stage[stage_col], forget_stage[max_f_col], marker='o')
ax.set_xlabel('Sequential Stage')
ax.set_ylabel('Maximum Forgetting')
ax.set_title('Maximum Task Forgetting Across Sequential Stages')
plt.tight_layout()
plt.savefig(FIGURES / 'max_forgetting_curve.png', bbox_inches='tight')
plt.show()

### 4.3 Sequential RMSE

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(forget_stage[stage_col], forget_stage[rmse_col], marker='o')
ax.set_xlabel('Sequential Stage')
ax.set_ylabel('RMSE')
ax.set_title('Overall RMSE During Sequential Training')
plt.tight_layout()
plt.savefig(FIGURES / 'sequential_rmse.png', bbox_inches='tight')
plt.show()

### 4.4 Number of forgotten tasks

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(forget_stage[stage_col], forget_stage[count_col], marker='o')
ax.set_xlabel('Sequential Stage')
ax.set_ylabel('Tasks with Positive Forgetting')
ax.set_title('Number of Forgotten Tasks Across Sequential Stages')
plt.tight_layout()
plt.savefig(FIGURES / 'forgotten_task_count.png', bbox_inches='tight')
plt.show()

## 5. Final forgetting summary

In [ ]:
final_stage = forget_stage.iloc[-1]
peak_mean_idx = forget_stage[mean_f_col].astype(float).idxmax()
peak_max_idx = forget_stage[max_f_col].astype(float).idxmax()

summary = pd.DataFrame({
    'Measure': [
        'Final Stage', 'Final Seen Tasks', 'Final RMSE',
        'Final Mean Forgetting', 'Final Max Forgetting',
        'Final Forgotten Tasks', 'Final Mean Degradation',
        'Peak Mean Forgetting Stage', 'Peak Mean Forgetting',
        'Peak Max Forgetting Stage', 'Peak Max Forgetting'
    ],
    'Value': [
        final_stage[stage_col], final_stage[seen_col], final_stage[rmse_col],
        final_stage[mean_f_col], final_stage[max_f_col], final_stage[count_col],
        final_stage[deg_col], forget_stage.loc[peak_mean_idx, stage_col],
        forget_stage.loc[peak_mean_idx, mean_f_col], forget_stage.loc[peak_max_idx, stage_col],
        forget_stage.loc[peak_max_idx, max_f_col]
    ]
})
display(summary)
summary.to_csv(RESULTS / 'final_forgetting_summary.csv', index=False)

## 6. Task-level forgetting analysis

In [ ]:
display(forget_tasks.head())

task_id_col = find_col(forget_tasks, ['task_id', 'task'])
forget_col = find_col(forget_tasks, ['forgetting', 'final_forgetting'])
best_col = find_col(forget_tasks, ['best_rmse', 'best rmse'])
final_task_rmse_col = find_col(forget_tasks, ['final_rmse', 'final rmse', 'current_rmse'])

print({'task_id': task_id_col, 'forgetting': forget_col,
       'best_rmse': best_col, 'final_rmse': final_task_rmse_col})

### Top 20 most-forgotten tasks

In [ ]:
if forget_col is not None:
    top20 = forget_tasks.sort_values(forget_col, ascending=False).head(20).copy()
    display(top20)
    top20.to_csv(RESULTS / 'top20_most_forgotten_tasks.csv', index=False)
else:
    print('A forgetting column was not detected. Check the printed column names above.')

### Forgetting distribution

In [ ]:
if forget_col is not None:
    values = pd.to_numeric(forget_tasks[forget_col], errors='coerce').dropna()
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.hist(values, bins=30)
    ax.axvline(0, linestyle='--', linewidth=1)
    ax.set_xlabel('Final Forgetting')
    ax.set_ylabel('Number of Tasks')
    ax.set_title('Distribution of Task-Level Forgetting')
    plt.tight_layout()
    plt.savefig(FIGURES / 'forgetting_distribution.png', bbox_inches='tight')
    plt.show()
else:
    print('A forgetting column was not detected.')

## 7. Final interpretation

Use the generated tables and figures rather than claiming that task embeddings improved every metric. The completed experiment showed that the task-embedding model should be compared metric-by-metric, while the forgetting experiment should be described as a sequential fine-tuning / forgetting stress test.

In [ ]:
print('FINAL RESULTS FILES CREATED')
print('- final_model_comparison.csv')
print('- final_forgetting_summary.csv')
print('- top20_most_forgotten_tasks.csv (if task forgetting column is detected)')
print('- figures/*.png')

print('\nFigure files:')
for f in sorted(FIGURES.glob('*.png')):
    print(' ', f.name)